# 01 - Circuit models

This notebook introduces the circuit models of `twpasolver` and how they combine into networks.

## Overview

Every component is a model class, derived from `twpasolver.twoport.TwoPortModel`, that computes its ABCD matrix on an array of frequencies. The models are in `twpasolver.models` and give their response through three methods:

* `get_abcd(freqs)`: an `ABCDArray`, a NumPy array of 2x2 matrices with fast matrix products, the building block for cascading components;
* `get_cell(freqs)`: a `TwoPortCell`, which holds the frequencies and the `ABCDArray` and converts them to S-parameters or to a `scikit-rf` `Network`;
* `get_network(freqs)`: the `scikit-rf` `Network` directly.

Every model can also be repeated `N` times in cascade (the `N` field), validates its parameters, and saves to and loads from JSON (through `pydantic`).

Parameters and frequencies are in SI units: H, F, $\Omega$ and Hz.

## Example: a capacitor

We create a capacitor, compute its response over a frequency range and plot $S_{11}$, both from the `scikit-rf` `Network` and from the `TwoPortCell`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from twpasolver.mathutils import to_dB
from twpasolver.models import Capacitance

# Define frequency range
freqs = np.arange(1e9, 10e9, 1e6)

# Create capacitor
C_val = 1e-12  # 1 pF
C = Capacitance(C=C_val)

# Get network response and plot S11
net = C.get_network(freqs)
net.s11.plot_s_db(label="S11, from skrf.Network", lw=3)

# Get cell response and plot S11
cell = C.get_cell(freqs)
plt.plot(cell.freqs, to_dB(cell.s.S11), label="S11, from TwoPortCell", ls="--", lw=3)
plt.ylabel("S11 (dB)")
plt.legend()
plt.show()

The model saves to JSON and loads back unchanged:

In [ ]:
C.dump_to_file("Capacitor.json")
C_recovered = Capacitance.from_file("Capacitor.json")
print(C == C_recovered)
C.model_dump()

## Model arrays

Models combine into arrays whose response is computed automatically, either by creating the arrays directly or with `twpasolver.models.compose`. `compose` returns:

* a `OnePortArray` when all elements are `OnePortModel`s with the same `twoport_parallel` setting (series or shunt placement in a two-port network). One-port elements are defined by their impedance alone, so they combine in series or in parallel into a new one-port element before entering a two-port network;
* a `TwoPortArray` otherwise, whose response is the product of the ABCD matrices of its elements.

Both kinds of array support list operations (appending, indexing, slicing), nesting, and JSON serialization of nested structures.

## Example: LC resonators

An inductor, a capacitor and a small resistance in series form an LC resonator, a `OnePortArray`; we plot its response.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from twpasolver.mathutils import to_dB
from twpasolver.models import Capacitance, Inductance, Resistance, compose

# Define frequency range
freqs = np.arange(1e9, 10e9, 2e6)

# Resonance frequency and inductance value
f_res = 5e9
L_val = 1e-9
R_val = 1e-3

# Create series LC resonator
L = Inductance(L=L_val)
C = Capacitance(C=1 / ((f_res * 2 * np.pi) ** 2 * L_val))
R = Resistance(R=R_val)
LCres = compose(L, C, R)

net = LCres.get_network(freqs)
net.s11.plot_s_db(lw=3)
plt.ylabel("S11 (dB)")
plt.show()

Finally, we place three LC resonators with different resonance frequencies as shunt elements (`twoport_parallel = True`) in a `TwoPortArray` and plot the combined transmission.

In [ ]:
from twpasolver.models import TwoPortArray

# Create array of LC resonators with different resonance frequencies
LCarray = TwoPortArray()
for f_res in np.linspace(2e9, 8e9, 3):
    C = Capacitance(C=1 / ((f_res * 2 * np.pi) ** 2 * L_val))
    LC = compose(L, C, R)
    LC.twoport_parallel = True
    LCarray.append(LC)

# Get network response and plot S21
net = LCarray.get_network(freqs)
net.s21.plot_s_db(lw=3, label="S21")
plt.ylabel("S21 (dB)")
plt.show()

A TWPA is built the same way: `twpasolver.models.TWPA` is a `TwoPortArray` of unit cells that also carries the parameters of the nonlinear response. Notebook 02 shows how to define one.